## 03. 시각화 프로토타입

KB부동산 대시보드의 코로플레스 지도와 시계열 차트를 노트북에서 먼저 확인합니다.

In [ ]:
import sys
from pathlib import Path

# 프로젝트 루트를 sys.path에 추가 (notebooks/ -> part08_kb_dashboard/ -> projects/ -> 프로젝트 루트)
PROJECT_ROOT = str(Path.cwd().resolve().parents[2])
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from IPython.display import HTML, display

from projects.part08_kb_dashboard.dashboard.constants import (
    INDICATORS,
    MAP_LAYOUT,
    SIDO_MAPPING,
)
from projects.part08_kb_dashboard.dashboard.geo import load_geojson, filter_by_sido
from projects.part08_kb_dashboard.dashboard.preprocessing import (
    get_all_indicators,
    get_timeseries,
    get_ts_date_range,
)
from projects.part08_kb_dashboard.dashboard.choropleth import create_choropleth
from projects.part08_kb_dashboard.dashboard.charts import (
    create_line_chart,
    create_heatmap_table,
)

### 1. 조회 조건 설정

대시보드 기본 화면과 비슷하게 서울·경기 지역의 2025년 9월 데이터를 확인합니다.

In [ ]:
target_year = 2025
target_month = 9
selected_sido = ["서울", "경기"]
selected_sido_full = [SIDO_MAPPING[name] for name in selected_sido]

geojson = load_geojson()
filtered_geojson = filter_by_sido(geojson, selected_sido_full)
all_data = get_all_indicators(target_year, target_month)

print(f"조회 기준: {target_year}년 {target_month}월")
print(f"선택 시도: {', '.join(selected_sido_full)}")
print(f"지도 feature 수: {len(filtered_geojson['features'])}개")
for name in MAP_LAYOUT:
    print(f"{name}: {len(all_data[name])}행")

### 2. 4개 지표 코로플레스 지도

대시보드의 2×2 지도에 들어가는 4개 지표를 하나씩 생성합니다.

In [ ]:
maps = {}

for indicator_name in MAP_LAYOUT:
    info = INDICATORS[indicator_name]
    df = all_data[indicator_name]

    if info["type"] == "count":
        value_col = "거래량"
        sig_cd_col = "sig_kor_nm"
        decimals = 0
    elif info["type"] == "direct":
        value_col = "전세가율"
        sig_cd_col = "sig_cd"
        decimals = 1
    else:
        value_col = "증감률"
        sig_cd_col = "sig_cd"
        decimals = 1

    title = f"{indicator_name} ({target_year}-{target_month:02d})"
    maps[indicator_name] = create_choropleth(
        filtered_geojson,
        df,
        value_col=value_col,
        sig_cd_col=sig_cd_col,
        title=title,
        unit=info["unit"],
        auto_fit=True,
    )

    display(HTML(f"<h4>{title}</h4>"))
    display(maps[indicator_name])

### 3. 지도 상세 프로토타입

지도 상세 뷰는 4개 지표 중 하나를 크게 확인하는 화면입니다. 여기서는 매매가격 전년동월대비 증감률을 확대해서 봅니다.

In [ ]:
detail_indicator = "매매 전년동월대비 증감률"
display(maps[detail_indicator])

### 4. 시계열 분석 데이터 준비

선택한 시군구의 매매가격 전년동월대비 증감률 추이를 조회합니다. 데이터가 있는 경우 강남구·송파구·마포구·성남분당구·용산구를 우선 선택합니다.

In [ ]:
ts_indicator = "매매 전년동월대비 증감률"
ts_info = INDICATORS[ts_indicator]

base_df = all_data[ts_indicator]
preferred_names = ["강남구", "송파구", "마포구", "성남분당구", "용산구"]
selected_regions = base_df[base_df["지역명"].isin(preferred_names)]

if selected_regions.empty:
    selected_regions = base_df.head(5)

sig_cds = selected_regions["sig_cd"].tolist()
region_names = selected_regions["지역명"].tolist()

ts_start_y, _, ts_end_y, _ = get_ts_date_range()
start_year = max(ts_start_y, target_year - 4)
end_year = min(ts_end_y, target_year)

ts_df = get_timeseries(
    table=ts_info["ts_table"],
    value_col=ts_info["value_col"],
    indicator_type=ts_info["type"],
    sig_cds=sig_cds,
    start_year=start_year,
    end_year=end_year,
)

print(f"선택 지역: {', '.join(region_names)}")
print(f"조회 기간: {start_year}년 ~ {end_year}년")
print(f"시계열 행 수: {len(ts_df)}행")
display(ts_df.head())

### 5. 선 그래프 확인

시군구별 전년동월대비 증감률을 한 차트에 겹쳐서 봅니다.

In [ ]:
fig = create_line_chart(
    ts_df,
    title=f"{ts_indicator} — {start_year}~{end_year}",
    unit=ts_info["unit"],
)
fig.show()

### 6. 히트맵 테이블 확인

같은 시계열 데이터를 표로 바꾸고 색상을 입히면, 어느 지역이 먼저 빨간색 또는 파란색으로 바뀌는지 비교하기 쉽습니다.

In [ ]:
styled = create_heatmap_table(ts_df, unit=ts_info["unit"])
display(styled)

### 7. HTML로 저장하기

필요하면 코로플레스 지도를 HTML 파일로 저장해서 브라우저에서 따로 열어볼 수 있습니다.

In [ ]:
output_path = Path("part08_choropleth_preview.html")
maps[detail_indicator].save(output_path)
print(f"지도가 {output_path}로 저장되었습니다.")